### Step 1: Import python libraries and api key

In [22]:
import os
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import display, Markdown
from openai import OpenAI
import gradio as gr

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if OPENAI_API_KEY is None:
    raise Exception("OPENAI_API_KEY environment variable is not set.")

### Step 2: Set up Pushover
* Step 2a: set up account in your browser (https://pushover.net)
* Step 2b: set up the app on your iPhone/Adroid phone, long into the same account
* Step 2c: in the browser create an "Application/API Toke"
* Step 2d: copy your User Key and API Token into the .env file like the following but without the dot 
* PUSHOVER_USER=XXXXXXXXXXXXXXXXX
* PUSHOVER_TOKEN=YYYYYYYYYYYYYYYYYYY

* Save changes to the .env file
* Run a test by manually sending a notification from Pushover in your browser to your phone. 

In [23]:
load_dotenv()

True

In [24]:
pushover_user=os.getenv("PUSHOVER_USER")
pushover_token=os.getenv("PUSHOVER_TOKEN")
pushover_url="https://api.pushover.net/1/messages.json"

In [25]:
# test Pushover
import requests

def send_notification(message: str):
    payload={"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [26]:
send_notification("Hello to myself from Pushover.")

### Step 3: Describe Pushover as an LLM tool

In [27]:
send_notification_function = {
    "name": "send_notification",
    "description": "Send a push notification to the user's phone via Pushover. User this to alert the user of time-sensitive issues.",
    "parameters": {
        "type": "object",
        "properties": {
            "message": {
                "type": "string",
                "description": "The notification message to send to the user's device"
            }
        },
        "required": ["message"]
    }
}

### Step 4: Add Pusherover to the list of tools for the LLM

In [28]:
tools = [{"type": "function", "function": send_notification_function}]

### Step 5: Call the tool from an LLM

In [ ]:
client=OpenAI()
response=client.chat.completions.create(
    model="gpt-4o-mini", 
    messages=[
        {"role": "user", "content": "Please tell me the amazing progress I have made in the ai engineering class, and send a message via Pushover to my phone."
        }
        ], 
        tools=tools
)

# Check if the model wants to call a tool
message = response.choices[0].message

In [30]:
print(message)

ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_stoWNOSho9TVUkVoGq1wG5dO', function=Function(arguments='{"message":"Congratulations on your amazing progress in the AI Engineering class! Your dedication and hard work are truly paying off. Keep pushing forward!"}', name='send_notification'), type='function')])


In [32]:
if message.tool_calls:
    tool_call=message.tool_calls[0]
    import json
    args=json.loads(tool_call.function.arguments)

    # Actually send the notification
    send_notification(args["message"])
    print(f"Send notification: {args["message"]}")

else: 
    print(message.content)



Send notification: Congratulations on your amazing progress in the AI Engineering class! Your dedication and hard work are truly paying off. Keep pushing forward!
